# CoreTech Lead Scoring System

This notebook builds a **lead scoring system (0–100)** for CoreTech Solutions, a B2B IT
services company. It loads a sample dataset of 60 leads, scores each lead using a weighted
rule-based model across five factors, and outputs:

- **lead_score** (0–100)
- **priority** (`High` / `Medium` / `Low`)
- **service_recommendation**
- **explanation** (short justification of the score)

**Sections:**
1. Setup & Load Data
2. Exploratory Data Overview
3. Scoring Logic (Budget, Timeline, Urgency, Company Size, Lead Source)
4. Apply Scoring to All Leads
5. Visualize Results
6. Export Final Scored Dataset


## 1. Setup & Load Data

The dataset is embedded in this notebook (see hidden cell below), so it's written to `coretech_leads.csv` automatically — **no file upload needed**. The same CSV is also included separately in this repo for reference/reuse outside the notebook.

In [ ]:
# This cell writes the embedded dataset to coretech_leads.csv so the notebook
# runs standalone in Colab/Jupyter with no manual file upload required.
csv_content = """lead_id,name,company,industry,email,phone,lead_source,company_size,budget_usd,timeline,urgency,interested_service,notes
L001,Sumaiya Malik,NextGen Retailers,Retail,sumaiya.malik@nextgenretaile.com,+92-323-4744854,Cold Call,201-500,137500,3-6 months,Low,Custom Software Development,"Downloaded whitepaper, opened all follow-up emails."
L002,Sara Khan,Skyline Logistics,Education,sara.khan@skylinelogisti.com,+92-313-9478454,Organic Search,1-10,9000,Short-term (1 month),Medium,IT Helpdesk Outsourcing,Decision maker confirmed budget internally.
L003,Maham Latif,BrightPath Edu,Agriculture,maham.latif@brightpathedu.com,+92-302-8090293,Webinar,11-50,11000,Short-term (1 month),Medium,Custom Software Development,"Downloaded whitepaper, opened all follow-up emails."
L004,Hina Aziz,Vertex Manufacturing,Textiles,hina.aziz@vertexmanufact.com,+92-344-1728977,Partner Referral,51-200,44000,6+ months / Exploring,High,IT Helpdesk Outsourcing,CFO involved in early conversations.
L005,Sumaiya Siddiqui,Coastal Foods Co,Textiles,sumaiya.siddiqui@coastalfoodsco.com,+92-343-2166941,Referral,201-500,173500,3-6 months,Low,Cybersecurity Audit,Requested a follow-up demo next week.
L006,Sadia Butt,PrimeHealth Clinics,Hospitality,sadia.butt@primehealthcli.com,+92-322-7210606,Webinar,11-50,10500,Short-term (1 month),Low,Network Setup & Support,Reached out directly asking for a quote.
L007,Sumaiya Sheikh,Urban Builders Group,Robotics,sumaiya.sheikh@urbanbuildersg.com,+92-312-8755439,Paid Ad,11-50,30500,6+ months / Exploring,Medium,Cybersecurity Audit,Referred by an existing client.
L008,Asim Ahmed,DataSphere Analytics,Technology,asim.ahmed@datasphereanal.com,+92-305-7730428,Trade Show,1-10,7000,6+ months / Exploring,Medium,AI/Automation Consulting,Decision maker confirmed budget internally.
L009,Bilawal Saeed,GreenLeaf Agro,Telecom,bilawal.saeed@greenleafagro.com,+92-332-5443951,Cold Call,11-50,22000,1-3 months,High,Mobile App Development,Wants a pilot project before full rollout.
L010,Kashif Farooq,Metro Transit Systems,Textiles,kashif.farooq@metrotransitsy.com,+92-312-9548432,Partner Referral,1-10,4500,Immediate (0-2 weeks),Critical,Network Setup & Support,"Early-stage research, no clear timeline yet."
L011,Danish Javed,Falcon Security Services,Media,danish.javed@falconsecurity.com,+92-306-7402509,Organic Search,500+,180000,1-3 months,Low,Cloud Migration,Attended webinar and asked about pricing tiers.
L012,Hira Anwar,Horizon Textiles,Agriculture,hira.anwar@horizontextile.com,+92-321-5924115,Paid Ad,11-50,16500,Immediate (0-2 weeks),Medium,AI/Automation Consulting,Mentioned internal approval still pending.
L013,Saira Sheikh,BlueWave Telecom,Real Estate,saira.sheikh@bluewaveteleco.com,+92-304-9517169,Organic Search,11-50,38000,1-3 months,High,IT Helpdesk Outsourcing,"Met at industry event, expressed strong interest."
L014,Anosha Mahmood,Apex Financial Group,Energy,anosha.mahmood@apexfinancialg.com,+92-301-7089806,Trade Show,11-50,11000,Short-term (1 month),Medium,Custom Software Development,Has urgent compliance deadline driving the project.
L015,Rizwan Raza,Silverline Hospitality,Robotics,rizwan.raza@silverlinehosp.com,+92-312-8973915,Email Campaign,11-50,38500,1-3 months,Medium,Cybersecurity Audit,Mentioned internal approval still pending.
L016,Saira Iqbal,Crescent Energy,Aviation,saira.iqbal@crescentenergy.com,+92-326-7264956,Partner Referral,500+,297500,1-3 months,High,Custom Software Development,"Met at industry event, expressed strong interest."
L017,Kashif Anwar,Pinnacle Real Estate,Technology,kashif.anwar@pinnaclereales.com,+92-343-1120642,Website Form,201-500,172500,Immediate (0-2 weeks),Critical,Cloud Migration,Referred by an existing client.
L018,Mariam Riaz,Quantum Robotics,Technology,mariam.riaz@quantumrobotic.com,+92-327-4594297,Email Campaign,11-50,16000,6+ months / Exploring,Medium,ERP Integration,"Early-stage research, no clear timeline yet."
L019,Aiza Javed,EverGreen Packaging,Construction,aiza.javed@evergreenpacka.com,+92-301-8231838,Webinar,51-200,68000,1-3 months,Medium,Network Setup & Support,Attended webinar and asked about pricing tiers.
L020,Hina Ahmed,Stellar Media House,Telecom,hina.ahmed@stellarmediaho.com,+92-321-5171761,LinkedIn,11-50,23000,1-3 months,High,Data Analytics Platform,Decision maker confirmed budget internally.
L021,Ahsan Raza,TrustBank Microfinance,Hospitality,ahsan.raza@trustbankmicro.com,+92-341-1848731,Email Campaign,201-500,82000,6+ months / Exploring,Low,Cybersecurity Audit,"Downloaded whitepaper, opened all follow-up emails."
L022,Rizwan Saeed,Bright Future School Network,Construction,rizwan.saeed@brightfuturesc.com,+92-330-3762152,Paid Ad,1-10,6500,1-3 months,Low,ERP Integration,"Downloaded whitepaper, opened all follow-up emails."
L023,Hafsa Anwar,Orbit E-commerce,Pharma,hafsa.anwar@orbite-commerc.com,+92-332-4185957,Trade Show,11-50,32500,Immediate (0-2 weeks),Medium,Cloud Migration,Referred by an existing client.
L024,Hamza Ahmed,Ironclad Manufacturing,Packaging,hamza.ahmed@ironcladmanufa.com,+92-338-9910817,Cold Call,1-10,11500,1-3 months,Low,Custom Software Development,Requested a follow-up demo next week.
L025,Danish Qureshi,CityLink Insurance,Telecom,danish.qureshi@citylinkinsura.com,+92-309-5130808,Organic Search,51-200,62000,3-6 months,High,Mobile App Development,Mentioned internal approval still pending.
L026,Mehak Butt,FreshMart Supermarkets,Construction,mehak.butt@freshmartsuper.com,+92-323-5456272,Paid Ad,11-50,8500,3-6 months,Medium,Custom Software Development,Asked detailed technical questions on call.
L027,Yasir Latif,NovaTech Software,Manufacturing,yasir.latif@novatechsoftwa.com,+92-308-4576135,Email Campaign,11-50,20000,6+ months / Exploring,Low,Cybersecurity Audit,"Currently using a competitor, exploring switch."
L028,Fatima Akhtar,Harvest Valley Farms,Robotics,fatima.akhtar@harvestvalleyf.com,+92-329-9874152,Referral,201-500,175000,1-3 months,Low,Custom Software Development,Comparing us against two other vendors.
L029,Rabia Malik,Royal Garments Ltd,Manufacturing,rabia.malik@royalgarmentsl.com,+92-342-5569244,Trade Show,201-500,172000,3-6 months,High,Network Setup & Support,"Currently using a competitor, exploring switch."
L030,Komal Saeed,ClearView Optics,Agriculture,komal.saeed@clearviewoptic.com,+92-301-8106422,Trade Show,1-10,4500,Short-term (1 month),Critical,Data Analytics Platform,Has urgent compliance deadline driving the project.
L031,Maham Anwar,Summit Construction,Aviation,maham.anwar@summitconstruc.com,+92-338-1162230,Website Form,1-10,2500,6+ months / Exploring,Low,IT Helpdesk Outsourcing,CFO involved in early conversations.
L032,Waqas Latif,Lighthouse Legal Services,Robotics,waqas.latif@lighthouselega.com,+92-316-3138181,Referral,51-200,84000,6+ months / Exploring,Medium,Cloud Migration,Referred by an existing client.
L033,Kamran Qureshi,RapidShip Couriers,Pharma,kamran.qureshi@rapidshipcouri.com,+92-305-7817881,Organic Search,201-500,165000,6+ months / Exploring,High,IT Infrastructure Support,"Early-stage research, no clear timeline yet."
L034,Saad Javed,GoldenAge Pharma,Retail,saad.javed@goldenagepharm.com,+92-315-7907400,LinkedIn,11-50,36000,3-6 months,Critical,Cloud Migration,Asked detailed technical questions on call.
L035,Sana Iqbal,BlueOcean Seafoods,Hospitality,sana.iqbal@blueoceanseafo.com,+92-324-4818418,LinkedIn,1-10,14500,Short-term (1 month),High,Custom Software Development,"Currently using a competitor, exploring switch."
L036,Areeba Riaz,Pioneer Auto Parts,Telecom,areeba.riaz@pioneerautopar.com,+92-345-1463060,Website Form,500+,140000,Short-term (1 month),Medium,Data Analytics Platform,Requested a follow-up demo next week.
L037,Anosha Javed,Crystal Hotels Group,Textiles,anosha.javed@crystalhotelsg.com,+92-326-9580255,Website Form,51-200,53500,1-3 months,High,AI/Automation Consulting,"Met at industry event, expressed strong interest."
L038,Adeel Anwar,Unity Healthcare,Pharma,adeel.anwar@unityhealthcar.com,+92-315-8235969,Website Form,500+,185500,6+ months / Exploring,Medium,Network Setup & Support,Has urgent compliance deadline driving the project.
L039,Babar Chaudhry,Falconwing Airlines,Real Estate,babar.chaudhry@falconwingairl.com,+92-326-6472441,Paid Ad,201-500,109500,1-3 months,High,Network Setup & Support,Attended webinar and asked about pricing tiers.
L040,Naveed Sheikh,TechNest Startups,Media,naveed.sheikh@techneststartu.com,+92-344-7812810,Email Campaign,201-500,139500,Short-term (1 month),High,Mobile App Development,Attended webinar and asked about pricing tiers.
L041,Mehak Akhtar,Maple Dairy Co,Hospitality,mehak.akhtar@mapledairyco.com,+92-333-9576076,Partner Referral,201-500,71000,6+ months / Exploring,Medium,Network Setup & Support,"Currently using a competitor, exploring switch."
L042,Komal Siddiqui,RedBrick Realty,Security,komal.siddiqui@redbrickrealty.com,+92-303-6207974,LinkedIn,201-500,140000,Immediate (0-2 weeks),Critical,ERP Integration,CFO involved in early conversations.
L043,Furqan Raza,Skyward Aviation,Hospitality,furqan.raza@skywardaviatio.com,+92-339-4262081,Paid Ad,51-200,83500,Short-term (1 month),Medium,Cloud Migration,"Early-stage research, no clear timeline yet."
L044,Ahsan Malik,GreenField Dairy,Finance,ahsan.malik@greenfielddair.com,+92-312-9689889,Partner Referral,1-10,4000,Short-term (1 month),Low,ERP Integration,"Early-stage research, no clear timeline yet."
L045,Salman Riaz,Vantage Consulting,Robotics,salman.riaz@vantageconsult.com,+92-345-8425149,Organic Search,201-500,118500,6+ months / Exploring,High,IT Helpdesk Outsourcing,Wants a pilot project before full rollout.
L046,Fatima Saeed,BrightSpark Electronics,Hospitality,fatima.saeed@brightsparkele.com,+92-323-5652512,Email Campaign,51-200,43500,Short-term (1 month),High,AI/Automation Consulting,Decision maker confirmed budget internally.
L047,Faraz Mahmood,Coral Bay Resorts,Security,faraz.mahmood@coralbayresort.com,+92-341-3321531,Cold Call,11-50,21500,3-6 months,Medium,Custom Software Development,"Downloaded whitepaper, opened all follow-up emails."
L048,Tariq Anwar,Northstar Logistics,Hospitality,tariq.anwar@northstarlogis.com,+92-330-4470105,Paid Ad,51-200,81000,3-6 months,Low,Cloud Migration,Wants a pilot project before full rollout.
L049,Saira Latif,PureWater Utilities,Telecom,saira.latif@purewaterutili.com,+92-330-6901533,Trade Show,201-500,130500,6+ months / Exploring,Low,Website & E-commerce Development,Has urgent compliance deadline driving the project.
L050,Naveed Anwar,Everest Trading Co,Media,naveed.anwar@everesttrading.com,+92-317-4681284,Trade Show,51-200,77500,Immediate (0-2 weeks),High,Network Setup & Support,"Downloaded whitepaper, opened all follow-up emails."
L051,Amna Sheikh,SunRise Solar Energy,Hospitality,amna.sheikh@sunrisesolaren.com,+92-319-9961107,Referral,500+,344500,1-3 months,Medium,Custom Software Development,Comparing us against two other vendors.
L052,Ahsan Akhtar,Citywide Pharmacy Chain,Healthcare,ahsan.akhtar@citywidepharma.com,+92-304-7360307,Webinar,11-50,17000,Short-term (1 month),Medium,Website & E-commerce Development,"Early-stage research, no clear timeline yet."
L053,Asim Javed,Blossom Bakery Chain,Agriculture,asim.javed@blossombakeryc.com,+92-307-1325197,Email Campaign,1-10,15000,6+ months / Exploring,High,Custom Software Development,Attended webinar and asked about pricing tiers.
L054,Asad Iqbal,IronGate Security,Retail,asad.iqbal@irongatesecuri.com,+92-342-5002120,Cold Call,51-200,71000,Immediate (0-2 weeks),Low,ERP Integration,"Currently using a competitor, exploring switch."
L055,Furqan Aziz,Velocity Sports Gear,Healthcare,furqan.aziz@velocitysports.com,+92-349-2921542,Cold Call,500+,317000,Immediate (0-2 weeks),Critical,Data Analytics Platform,"Downloaded whitepaper, opened all follow-up emails."
L056,Shahzad Iqbal,Cedar Valley Furniture,Education,shahzad.iqbal@cedarvalleyfur.com,+92-343-2709620,Trade Show,500+,141500,3-6 months,Critical,Mobile App Development,Referred by an existing client.
L057,Shazia Javed,Hilltop Dairy Farms,Pharma,shazia.javed@hilltopdairyfa.com,+92-321-9488613,Webinar,1-10,8000,1-3 months,Medium,Website & E-commerce Development,Attended webinar and asked about pricing tiers.
L058,Babar Akhtar,Quantum Leap Education,Aviation,babar.akhtar@quantumleapedu.com,+92-316-3955087,Email Campaign,500+,373500,Short-term (1 month),Low,IT Helpdesk Outsourcing,Asked detailed technical questions on call.
L059,Junaid Latif,TrueNorth Insurance,Agriculture,junaid.latif@truenorthinsur.com,+92-323-2453962,Trade Show,500+,318500,Short-term (1 month),Medium,Mobile App Development,Referred by an existing client.
L060,Bilal Saeed,Patriot Defense Supplies,Security,bilal.saeed@patriotdefense.com,+92-317-4558780,Webinar,201-500,175500,Short-term (1 month),Low,AI/Automation Consulting,"Currently using a competitor, exploring switch."
"""

with open("coretech_leads.csv", "w", encoding="utf-8") as f:
    f.write(csv_content)

print("coretech_leads.csv written to disk.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("coretech_leads.csv")
print(f"Loaded {len(df)} leads with {df.shape[1]} columns")
df.head()


## 2. Exploratory Data Overview

Quick look at the raw lead data before scoring.

In [ ]:
df.info()


In [ ]:
df.describe(include='all').T


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

df['lead_source'].value_counts().plot(kind='bar', ax=axes[0,0], title='Leads by Source', color='#4C72B0')
df['company_size'].value_counts().plot(kind='bar', ax=axes[0,1], title='Leads by Company Size', color='#55A868')
df['urgency'].value_counts().plot(kind='bar', ax=axes[1,0], title='Leads by Urgency', color='#C44E52')
df['timeline'].value_counts().plot(kind='bar', ax=axes[1,1], title='Leads by Timeline', color='#8172B2')

for ax in axes.flat:
    ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.show()


## 3. Scoring Logic

The model assigns points across **five weighted factors**, totaling 100 points:

| Factor | Max Points | Rationale |
|---|---|---|
| Budget | 30 | Strongest predictor of deal value and ability to close |
| Timeline | 25 | Faster timelines mean higher likelihood of near-term conversion |
| Urgency | 20 | Captures the lead's stated pain/urgency level |
| Company Size | 15 | Larger companies generally mean bigger, more stable contracts |
| Lead Source | 10 | Referrals/trade shows convert better than cold outreach |

Each factor uses a lookup table (bands for budget, dictionaries for categorical fields)
so the weights and cutoffs are easy to tune.

In [ ]:
BUDGET_BANDS = [
    (100000, None, 30),
    (50000, 99999, 25),
    (20000, 49999, 18),
    (10000, 19999, 12),
    (5000, 9999, 6),
    (0, 4999, 2),
]

TIMELINE_SCORES = {
    "Immediate (0-2 weeks)": 25,
    "Short-term (1 month)": 20,
    "1-3 months": 14,
    "3-6 months": 8,
    "6+ months / Exploring": 3,
}

URGENCY_SCORES = {
    "Critical": 20,
    "High": 15,
    "Medium": 9,
    "Low": 3,
}

COMPANY_SIZE_SCORES = {
    "500+": 15,
    "201-500": 12,
    "51-200": 9,
    "11-50": 5,
    "1-10": 2,
}

LEAD_SOURCE_SCORES = {
    "Referral": 10,
    "Partner Referral": 10,
    "Trade Show": 8,
    "Webinar": 7,
    "Website Form": 6,
    "LinkedIn": 6,
    "Organic Search": 5,
    "Email Campaign": 4,
    "Paid Ad": 3,
    "Cold Call": 2,
}

PRIORITY_THRESHOLDS = [
    (70, 100, "High"),
    (40, 69, "Medium"),
    (0, 39, "Low"),
]

SERVICE_CATALOG = {
    "Cloud Migration": "Cloud Migration & Managed Hosting Package",
    "Custom Software Development": "Custom Software Development Engagement",
    "IT Infrastructure Support": "Managed IT Infrastructure Support Plan",
    "Cybersecurity Audit": "Cybersecurity Audit & Hardening Package",
    "Data Analytics Platform": "Data Analytics & BI Platform Setup",
    "CRM Implementation": "CRM Implementation & Integration",
    "Website & E-commerce Development": "Website / E-commerce Development Package",
    "ERP Integration": "ERP Integration & Workflow Automation",
    "IT Helpdesk Outsourcing": "Outsourced IT Helpdesk Plan",
    "Mobile App Development": "Mobile App Development Engagement",
    "Network Setup & Support": "Network Setup & Support Package",
    "AI/Automation Consulting": "AI & Automation Consulting Engagement",
}


In [ ]:
def score_budget(budget):
    for lo, hi, pts in BUDGET_BANDS:
        if hi is None and budget >= lo:
            return pts
        if hi is not None and lo <= budget <= hi:
            return pts
    return 0

def score_timeline(timeline):
    return TIMELINE_SCORES.get(str(timeline).strip(), 0)

def score_urgency(urgency):
    return URGENCY_SCORES.get(str(urgency).strip(), 0)

def score_company_size(size):
    return COMPANY_SIZE_SCORES.get(str(size).strip(), 0)

def score_lead_source(source):
    return LEAD_SOURCE_SCORES.get(str(source).strip(), 0)

def get_priority(score):
    for lo, hi, label in PRIORITY_THRESHOLDS:
        if lo <= score <= hi:
            return label
    return "Low"

def recommend_service(interested_service, score):
    base = SERVICE_CATALOG.get(str(interested_service).strip(), "General IT Consulting Package")
    if score >= 70:
        return f"{base} (Enterprise/Priority Track)"
    return base

def build_explanation(row, b, t, u, c, s, total, priority):
    drivers = []
    if b >= 25:
        drivers.append(f"strong budget (${row['budget_usd']:,.0f})")
    elif b <= 6:
        drivers.append(f"low budget (${row['budget_usd']:,.0f})")
    if t >= 20:
        drivers.append(f"fast timeline ({row['timeline']})")
    elif t <= 8:
        drivers.append(f"long/unclear timeline ({row['timeline']})")
    if u >= 15:
        drivers.append(f"{row['urgency'].lower()} urgency")
    elif u <= 3:
        drivers.append("low urgency")
    if c >= 12:
        drivers.append(f"large company ({row['company_size']} employees)")
    elif c <= 2:
        drivers.append(f"very small company ({row['company_size']} employees)")
    if s >= 8:
        drivers.append(f"high-quality source ({row['lead_source']})")
    elif s <= 3:
        drivers.append(f"low-intent source ({row['lead_source']})")
    if not drivers:
        drivers.append("balanced, average signals across all factors")
    return f"Score {total}/100 ({priority} priority) - driven by " + "; ".join(drivers) + "."


## 4. Apply Scoring to All Leads

In [ ]:
def score_leads(df):
    df = df.copy()
    budget_pts = df["budget_usd"].apply(score_budget)
    timeline_pts = df["timeline"].apply(score_timeline)
    urgency_pts = df["urgency"].apply(score_urgency)
    size_pts = df["company_size"].apply(score_company_size)
    source_pts = df["lead_source"].apply(score_lead_source)

    df["budget_score"] = budget_pts
    df["timeline_score"] = timeline_pts
    df["urgency_score"] = urgency_pts
    df["company_size_score"] = size_pts
    df["lead_source_score"] = source_pts

    df["lead_score"] = (budget_pts + timeline_pts + urgency_pts + size_pts + source_pts).clip(0, 100)
    df["priority"] = df["lead_score"].apply(get_priority)

    df["service_recommendation"] = [
        recommend_service(row["interested_service"], row["lead_score"])
        for _, row in df.iterrows()
    ]
    df["explanation"] = [
        build_explanation(row, row["budget_score"], row["timeline_score"], row["urgency_score"],
                           row["company_size_score"], row["lead_source_score"],
                           row["lead_score"], row["priority"])
        for _, row in df.iterrows()
    ]
    return df

scored_df = score_leads(df)
scored_df = scored_df.sort_values("lead_score", ascending=False).reset_index(drop=True)
scored_df[["lead_id", "company", "lead_score", "priority", "service_recommendation"]].head(10)


In [ ]:
scored_df.loc[0, ["lead_id", "company", "lead_score", "priority",
                  "service_recommendation", "explanation"]]


## 5. Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

priority_counts = scored_df["priority"].value_counts().reindex(["High", "Medium", "Low"])
priority_counts.plot(kind="bar", ax=axes[0], color=["#C44E52", "#DD8452", "#55A868"])
axes[0].set_title("Leads by Priority")
axes[0].set_ylabel("Number of Leads")

axes[1].hist(scored_df["lead_score"], bins=15, color="#4C72B0", edgecolor="white")
axes[1].set_title("Lead Score Distribution")
axes[1].set_xlabel("Lead Score")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print("Priority breakdown:")
print(priority_counts.to_string())
print(f"\nAverage lead score: {scored_df['lead_score'].mean():.1f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
avg_components = scored_df[["budget_score", "timeline_score", "urgency_score",
                             "company_size_score", "lead_source_score"]].mean()
avg_components.plot(kind="bar", ax=ax, color="#8172B2")
ax.set_title("Average Score Contribution by Factor")
ax.set_ylabel("Average Points")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 6. Export Final Scored Dataset

Saves the final enriched dataset with all scoring columns to `scored_leads.csv`.

In [ ]:
scored_df.to_csv("scored_leads.csv", index=False)
print("Saved scored_leads.csv")
scored_df.head(10)


In [ ]:
# Quick lookup: view the full record (with explanation) for any single lead
lead_id_to_check = "L017"
scored_df[scored_df["lead_id"] == lead_id_to_check][
    ["lead_id", "company", "budget_usd", "timeline", "urgency", "company_size",
     "lead_source", "lead_score", "priority", "service_recommendation", "explanation"]
].T


---
### Notes on the model

- The scoring weights and thresholds (in Section 3) are business rules and can be tuned
  by editing the dictionaries/bands — no other code needs to change.
- This is a **rule-based / heuristic** scoring system, ideal for early-stage lead
  qualification where labeled historical conversion data isn't yet available.
- If historical win/loss data becomes available, this can be extended into a
  supervised ML model (e.g. logistic regression) using these same features.